# ASR Model Evaluation Notebook

Evaluate fine-tuned Whisper ASR models (LoRA adapters) using multiple metrics:
- **WER** (Word Error Rate)
- **CER** (Character Error Rate)  
- **MER** (Match Error Rate)
- **WIL** (Word Information Lost)

Results are saved to CSV and optionally logged to Weights & Biases.

## 1. Setup & Imports

In [ ]:
!pip install jiwer pandas huggingface_hub datasets transformers peft wandb torchcodec==0.7

In [ ]:
import os
import torch
import pandas as pd
from datetime import datetime
from tqdm.auto import tqdm
from datasets import load_dataset, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel
from jiwer import wer, cer, mer, wil

# Optional: Weights & Biases
try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    WANDB_AVAILABLE = False
    print("wandb not installed. Results will only be saved to CSV.")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
login(token="hf_token")
wandb.login(key="wandb_token")

## 2. Configuration

Configure your model and datasets here.

In [ ]:
# =============================================================================
# MODEL CONFIGURATION (LoRA Adapter)
# =============================================================================
BASE_MODEL_ID = "openai/whisper-small"  # Base Whisper model
ADAPTER_ID = "SPEAK-ASR/speak-whisper-small-si"  # Your LoRA adapter on HuggingFace
LANGUAGE = "sinhala"
TASK = "transcribe"

# =============================================================================
# DATASET CONFIGURATION
# =============================================================================
# List of datasets to evaluate (each evaluated separately for diversity analysis)
# Format: (dataset_name, split, num_samples or None for all)
DATASETS = [
    ("SPEAK-ASR/openslr-sinhala-asr-1", "test", 100),  # Use 100 samples
    ("SPEAK-ASR/openslr-sinhala-asr-2", "test", 100),
    # Add more datasets as needed:
    # ("your-dataset-name", "split", num_samples),
]

# Column names (same structure for all datasets)
AUDIO_COLUMN = "audio"
TEXT_COLUMN = "text"
SAMPLE_RATE = 16000

# =============================================================================
# OUTPUT CONFIGURATION  
# =============================================================================
OUTPUT_DIR = "../evaluation_results"
USE_WANDB = True  # Set to False to skip W&B logging
WANDB_PROJECT = "asr-evaluation"
WANDB_ENTITY = "SPEAK-ASR-uom"  # Your W&B entity/team name, or None for default

## 3. Load Model (LoRA Adapter)

In [ ]:
print(f"Loading base model: {BASE_MODEL_ID}")
print(f"Loading LoRA adapter: {ADAPTER_ID}")

# Load processor
processor = WhisperProcessor.from_pretrained(BASE_MODEL_ID)

# Load base model
base_model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL_ID)

# Load LoRA adapter
model = PeftModel.from_pretrained(base_model, ADAPTER_ID)
model = model.to(device)
model.eval()

# Set forced decoder IDs for language and task
forced_decoder_ids = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)

print(f"Model loaded successfully on {device}")

## 4. Evaluation Functions

In [ ]:
def transcribe_batch(audio_arrays: list, sampling_rate: int) -> list[str]:
    inputs = processor(
        audio_arrays,
        sampling_rate=sampling_rate,
        return_tensors="pt",
        padding=True
    ).input_features.to(device)

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features=inputs,
            task="transcribe",
            language="si",
        )

    return processor.batch_decode(predicted_ids, skip_special_tokens=True)

def compute_metrics(references: list, predictions: list) -> dict:
    """Compute all ASR metrics."""
    # Filter out empty strings
    valid_pairs = [(r, p) for r, p in zip(references, predictions) if r.strip()]
    if not valid_pairs:
        return {"wer": 0, "cer": 0, "mer": 0, "wil": 0}

    refs, preds = zip(*valid_pairs)
    refs, preds = list(refs), list(preds)

    return {
        "wer": wer(refs, preds),
        "cer": cer(refs, preds),
        "mer": mer(refs, preds),
        "wil": wil(refs, preds),
    }


def evaluate_dataset(dataset_name: str, split: str, num_samples: int = None) -> dict:
    """Evaluate model on a single dataset."""
    print(f"\n{'='*60}")
    print(f"Evaluating: {dataset_name} ({split})")
    print(f"{'='*60}")

    # Load dataset
    dataset = load_dataset(dataset_name, split=split)
    dataset = dataset.cast_column(AUDIO_COLUMN, Audio(sampling_rate=SAMPLE_RATE))

    # Sample if needed
    if num_samples and num_samples < len(dataset):
        dataset = dataset.select(range(len(dataset) - num_samples, len(dataset)))

    print(f"Samples to evaluate: {len(dataset)}")

    # Collect predictions
    references = []
    predictions = []

    BATCH_SIZE = 16   # try 16 / 32 on A6000
    for i in tqdm(range(0, len(dataset), BATCH_SIZE), desc="Transcribing"):
        batch = dataset[i:i+BATCH_SIZE]
    
        audio_arrays = [a["array"] for a in batch[AUDIO_COLUMN]]
        references.extend(batch[TEXT_COLUMN])
    
        try:
            preds = transcribe_batch(audio_arrays, SAMPLE_RATE)
        except Exception as e:
            print("Batch failed, skipping:", e)
            continue
    
        predictions.extend([p.strip() for p in preds])

    # Compute metrics
    metrics = compute_metrics(references, predictions)

    # Add metadata
    result = {
        "dataset": dataset_name,
        "split": split,
        "num_samples": len(dataset),
        **metrics
    }

    print(f"\nResults:")
    print(f"  WER: {metrics['wer']:.4f} ({metrics['wer']*100:.2f}%)")
    print(f"  CER: {metrics['cer']:.4f} ({metrics['cer']*100:.2f}%)")
    print(f"  MER: {metrics['mer']:.4f} ({metrics['mer']*100:.2f}%)")
    print(f"  WIL: {metrics['wil']:.4f} ({metrics['wil']*100:.2f}%)")

    return result, references, predictions

## 5. Run Evaluation

In [ ]:
# Initialize W&B if enabled
if USE_WANDB and WANDB_AVAILABLE:
    wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=f"eval-{ADAPTER_ID.split('/')[-1]}-{datetime.now().strftime('%Y%m%d_%H%M%S')}",
        config={
            "base_model_id": BASE_MODEL_ID,
            "adapter_id": ADAPTER_ID,
            "language": LANGUAGE,
            "task": TASK,
            "datasets": [d[0] for d in DATASETS],
        }
    )

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Run evaluation on all datasets
all_results = []
all_predictions = {}  # Store predictions for analysis

for dataset_name, split, num_samples in DATASETS:
    result, refs, preds = evaluate_dataset(dataset_name, split, num_samples)
    all_results.append(result)
    all_predictions[dataset_name] = {"references": refs, "predictions": preds}

    # Log to W&B
    if USE_WANDB and WANDB_AVAILABLE:
        wandb.log({
            f"{dataset_name.split('/')[-1]}/wer": result["wer"],
            f"{dataset_name.split('/')[-1]}/cer": result["cer"],
            f"{dataset_name.split('/')[-1]}/mer": result["mer"],
            f"{dataset_name.split('/')[-1]}/wil": result["wil"],
        })

## 6. Results Summary & Export

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame(all_results)
adapter_short_name = ADAPTER_ID.split("/")[-1]
csv_path_results = os.path.join(OUTPUT_DIR, f"eval_{adapter_short_name}_results.csv")
results_df.to_csv(csv_path_results, index=False)

# Calculate overall averages
avg_metrics = {
    "dataset": "AVERAGE",
    "split": "-",
    "num_samples": results_df["num_samples"].sum(),
    "wer": results_df["wer"].mean(),
    "cer": results_df["cer"].mean(),
    "mer": results_df["mer"].mean(),
    "wil": results_df["wil"].mean(),
}
results_df = pd.concat([results_df, pd.DataFrame([avg_metrics])], ignore_index=True)

# Display results
print("\n" + "="*80)
print("EVALUATION SUMMARY")
print("="*80)
print(f"Base Model: {BASE_MODEL_ID}")
print(f"LoRA Adapter: {ADAPTER_ID}")
print(f"\n{results_df.to_string(index=False)}")

# Save to CSV
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = os.path.join(OUTPUT_DIR, f"eval_{adapter_short_name}_{timestamp}.csv")
results_df.to_csv(csv_path, index=False)
print(f"\nResults saved to: {csv_path}")

# Log summary to W&B
if USE_WANDB and WANDB_AVAILABLE:
    wandb.log({
        "avg_wer": avg_metrics["wer"],
        "avg_cer": avg_metrics["cer"],
        "avg_mer": avg_metrics["mer"],
        "avg_wil": avg_metrics["wil"],
    })
    wandb.log({"results_table": wandb.Table(dataframe=results_df)})
    wandb.finish()
    print("Results logged to Weights & Biases")

## 7. Sample Predictions (Optional)

View some sample predictions for qualitative analysis.

In [ ]:
# Show sample predictions from first dataset
if all_predictions:
    first_dataset = list(all_predictions.keys())[0]
    refs = all_predictions[first_dataset]["references"]
    preds = all_predictions[first_dataset]["predictions"]
    
    print(f"Sample predictions from: {first_dataset}\n")
    for i in range(min(5, len(refs))):
        print(f"[{i+1}] Reference:  {refs[i]}")
        print(f"    Prediction: {preds[i]}")
        print()